*[cell 0]*

# LVL_1 Report — participants' sessions

Draws chosen participants' sessions from a finished LVL_1 handoff: the three LVL_1 detection features over the trials of
the participant's sessions (observed values only; NaN = gap), then the blocks of every block method of the handoff, one
strip per method. Use it to see whether the blocks capture the attention shifts suspected on manual review.
- **Dynamic segmentation:** the methods are the detectors, ordered as the LVL1 notebook ranks them (best first), and the
  smoothing the detectors apply is overlaid as they apply it (within each session, on observed stimuli only).
- **Predefined segmentation** (the handoff's `_meta.json` says which): the one method is `Predefined`, the blocks the
  data gives; no smoothing is drawn, as no detector runs.

**Run order.** After the LVL1 notebook (it reads the handoff and the detector summary that notebook writes). It writes
nothing.

**Inputs** (top of `[cell 1]`): `LVL1_HANDOFF`, the handoff prefix in `LVL_OUTS` (no default); `SWITCHERS`, the
participant ids to draw (`[]` = none); `METHODS`, the block methods to draw (`None` = every block method, the `Fixed_*`
matched controls excluded); `PROJECT_ROOT`, the project folder (README, "Paths"). The settings check stops the report
while `SWITCHERS` or `PROJECT_ROOT` holds `"SET_ME"` (README, "Settings"). The smoothing and feature definitions come
from the LVL1 notebook's detection-methods cell, through the upstream-definitions cell that `build_level_scripts.py`
writes.


In [ ]:
# [cell 1]
# ===== INPUTS: set these before running =====
LVL1_HANDOFF = None   # the LVL_1 handoff prefix in LVL_OUTS, e.g. "lv1_out_20261005_120000"
# participant ids to draw (e.g. suspected attention shifts on manual review); [] = none; EMST 30 ids, 2077, 4014, ...
SWITCHERS = "SET_ME"
# Block methods to visualize. None = every block method of the handoff (the Fixed_* matched controls are excluded):
# in dynamic segmentation every registered detector, ordered as the LVL1 notebook ranks them; in predefined
# segmentation "Predefined". Or hand-pick a list to keep the figure short.
METHODS = None

import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---- locations: everything is inside PROJECT_ROOT ----------------
# the absolute project folder holding code/, dset/ and dervatives/ (README, "Paths"; EMST /home/sjsuss626/EMST_files/sjs_dissertation)
PROJECT_ROOT = "SET_ME"
_SET_ME = {"PROJECT_ROOT": (PROJECT_ROOT, 'the absolute project folder holding code/, dset/ and dervatives/ (README, '
                                          '"Paths"); EMST /home/sjsuss626/EMST_files/sjs_dissertation'),
           "SWITCHERS": (SWITCHERS, "participant ids to draw (e.g. suspected attention shifts on manual review); "
                                    "[] = none; EMST 30 ids, 2077, 4014, ...")}   # name: (value, what to set)
_unset = [k for k, (v, _) in _SET_ME.items() if isinstance(v, str) and v == "SET_ME"]
if _unset:
    raise SystemExit(f'settings check (LVL1_report.ipynb): {len(_unset)} setting(s) still hold "SET_ME". Set each, then '
                     'run again (README, "Settings"):\n' + "\n".join(f"  - {k}  ([cell 1])\n      {_SET_ME[k][1]}"
                                                                    for k in _unset))
_OUT = Path(PROJECT_ROOT) / "dervatives" / "practical_application_generic"
LVL_OUTS = _OUT / "lvl_outs"

if LVL1_HANDOFF is None:
    raise SystemExit("set LVL1_HANDOFF (a LVL_1 handoff prefix in LVL_OUTS) at the top of this cell")
HANDOFF = LVL_OUTS / str(LVL1_HANDOFF).removesuffix("_long.csv")
_need = [Path(f"{HANDOFF}{s}") for s in ("_features.csv", "_meta.json", "_report_method_summary.csv")]
_missing = [str(p) for p in _need if not p.exists()]
if _missing:
    raise SystemExit("inputs not found (the LVL1 notebook writes them; README, \"Run order\"):\n  "
                     + "\n  ".join(_missing))


def _literal(v):
    """A handoff's _meta.json stores the LVL_1 configuration as str(value); read each back as its Python value."""
    try:
        return ast.literal_eval(v)
    except (ValueError, SyntaxError):
        return v


META = json.load(open(f"{HANDOFF}_meta.json"))
if META.get("segmentation") not in ("dynamic", "predefined"):
    raise SystemExit(f"{HANDOFF.name}_meta.json records segmentation {META.get('segmentation')!r}: use a handoff "
                     "written by this pipeline's LVL_1")
PREDEFINED = META["segmentation"] == "predefined"
LVL_1_CONFIG = {k: _literal(v) for k, v in META["lvl1_config"].items()}
summary_df = pd.read_csv(f"{HANDOFF}_report_method_summary.csv")   # every block method, best first
features_df = pd.read_csv(f"{HANDOFF}_features.csv")
print(f"LVL_1 handoff {HANDOFF.name}: {features_df['id'].nunique()} participants, "
      f"{features_df.groupby(['id', 'session']).ngroups} sessions; "
      + (f"predefined segmentation, blocks from column {META['predefined_block_col']!r} (no smoothing)" if PREDEFINED
         else f"smoothing {LVL_1_CONFIG['smoothing_method']}, window {LVL_1_CONFIG['smoothing_window']}"))

# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))


In [ ]:
# [notebook only] UPSTREAM DEFINITIONS (written by build_level_scripts.py)
# The definitions LVL1_report needs from the LVL_1 detection-methods cell ([cell 15]): ['FEATURE_REGISTRY', 'apply_smoothing', 'get_feature_column'].
# Edit them in the LVL1 notebook and rebuild.

FEATURE_REGISTRY = {
    "Adj RT Level (Stim)": {"rt_col": "aggregated_rt",       "residual_col": "aggregated_residual",      "direction": -1, "defined_on": "adj_rt_level_stim",
                        "validation_prefix": "adj_rt_level_stim"},
    "RT Variance":     {"rt_col": "rt_variance",         "residual_col": "residual_variance",        "direction": +1, "defined_on": "variance",
                        "validation_prefix": "rt_var"},
    # Temporal (sliding-window) RT variance: each scale's variance over a
    # derivative_filter_size window (centred when rolling_center=True), averaged across
    # scales (temporal_var_method="per_scale") -- the Esterman variance-time-course idea.
    # Distinct from "RT Variance" (cross-scale, within-stimulus).
    "Adj RT Var (win)":    {"rt_col": "local_rt_var",        "residual_col": "local_residual_var",       "direction": +1, "defined_on": "variance",
                        "validation_prefix": "adj_rt_var_win"},
    # Coherence (RT): mean pairwise distance of RTs across scales for a given
    # stimulus (instantaneous, cross-scale). High distance = low cross-scale
    # agreement = MORE inattentive.
    "Coherence (RT)":  {"rt_col": "coherence_rt",  "residual_col": "coherence_residual",      "direction": +1, "defined_on": "coherence",
                        "validation_prefix": "coherence_rt"},
    # Coherence (Var): pairwise distance of each scale's rolling RT variance
    # (cross-scale dispersion of the local variance signal, not a single scale's
    # own variance). High distance = MORE inattentive.
    "Coherence (Var)": {"rt_col": "coherence_vars", "residual_col": "coherence_residual_vars", "direction": +1, "defined_on": "coherence",
                        "validation_prefix": "coherence_var"},
    "Entropy (win)":       {"rt_col": "rating_entropy_norm", "residual_col": "rating_entropy_norm",      "direction": +1, "defined_on": "entropy",
                        "validation_prefix": "entropy_win"},
    "Rating Variance": {"rt_col": "rating_variance",     "residual_col": "rating_variance",          "direction": +1, "defined_on": "variance",
                        "validation_prefix": "rate_var"},
}


def get_feature_column(display_name, features, config):
    """
    Map feature display name to actual column name based on use_residuals setting.

    Reads from FEATURE_REGISTRY (single source of truth, defined above).

    Parameters
    ----------
    display_name : str
        Must be a key of FEATURE_REGISTRY (e.g. "Adj RT Level (Stim)", "RT Variance",
        "Coherence (RT)", "Coherence (Var)", "Entropy (win)", "Rating Variance").
    features : pd.DataFrame
        Feature dataframe to check column availability.
    config : dict
        Config with use_residuals setting.

    Returns
    -------
    str or None
        Column name if available, None otherwise. When use_residuals=True and
        the residual column is missing, the RT column is returned.
    """
    spec = FEATURE_REGISTRY.get(display_name)
    if spec is None:
        return None
    use_residuals = config["use_residuals"]
    primary = spec["residual_col"] if use_residuals else spec["rt_col"]
    if primary in features.columns:
        return primary
    # When the residual column is missing (e.g. the drift fit failed for this group), the RT column is used. With
    # use_residuals=False a missing RT column means something is wrong upstream, so None is returned.
    if use_residuals:
        rt_col = spec["rt_col"]
        if rt_col in features.columns:
            return rt_col
    return None


def apply_smoothing(signal, method, window):
    """
    Apply smoothing to a signal before changepoint detection.
    
    Parameters
    ----------
    signal : np.ndarray
        Input signal
    method : str
        "none", "boxcar", or "gaussian"
    window : int
        Smoothing window size
    
    Returns
    -------
    np.ndarray
        Smoothed signal
    """
    # explicit guard against invalid window sizes.
    if window <= 0:
        raise ValueError(f"smoothing_window must be > 0, got {window}")
    if method == "none" or window <= 1:
        return signal
    
    signal = np.asarray(signal, dtype=float)
    
    # NaN guard for other callers (e.g. plotting). The detectors never pass NaN: they
    # receive observed stimuli only (_on_observed_rows). np.convolve would spread a NaN
    # across the whole window, so any NaN is nearest-filled here before smoothing.
    if np.any(np.isnan(signal)):
        signal = pd.Series(signal).ffill().bfill().values
        if np.any(np.isnan(signal)):
            # All-NaN input: return as-is (cannot smooth nothing).
            return signal
    
    if method == "boxcar":
        kernel = np.ones(window) / window
    elif method == "gaussian":
        sigma = window / 6
        x = np.arange(window) - window // 2
        kernel = np.exp(-0.5 * (x / sigma) ** 2)
        kernel = kernel / kernel.sum()
    else:
        # Unknown smoothing method (typo? new value?). Warn rather than
        # silently return the unsmoothed signal -- otherwise a misspelt
        # LVL_1_CONFIG["smoothing_method"] disables smoothing without any signal.
        import warnings as _w_sm
        _w_sm.warn(
            f"apply_smoothing: unknown method '{method}'; expected one of "
            f"'none', 'boxcar', 'gaussian'. Returning signal unchanged.",
            stacklevel=2,
        )
        return signal
    
    # symmetric padding: avoids a one-element phase bias on even windows.
    pad_left = window // 2
    pad_right = (window - 1) // 2
    padded = np.pad(signal, (pad_left, pad_right), mode='reflect')
    smoothed = np.convolve(padded, kernel, mode='valid')
    
    if len(smoothed) > len(signal):
        smoothed = smoothed[:len(signal)]
    elif len(smoothed) < len(signal):
        smoothed = np.pad(smoothed, (0, len(signal) - len(smoothed)), mode='edge')
    
    return smoothed


In [ ]:
# [cell 2]
# =============================================================================
# PARTICIPANTS' SESSIONS: FEATURES, THE DETECTORS' SMOOTHING AND EVERY DETECTOR'S BLOCKS
# =============================================================================
# Per participant in SWITCHERS: the three LVL_1 detection features over the participant's sessions (NaN = gap), with
# the detectors' smoothing overlaid in dynamic segmentation (none in predefined segmentation: no detector smooths), then
# one strip per block method with its blocks (block_<m>_unique is used when present, block_<m> otherwise). Draws only.

# Get block columns available
block_cols = [c for c in features_df.columns if c.startswith('block_') and not c.endswith('_unique')]
print(f"\nBlock methods available: {len(block_cols)}")

# If nothing specified, skip the rest of the analysis
if not SWITCHERS:
    print("\nSWITCHERS is None or empty; skipping switcher analysis.")
else:
    # Filter to switchers only
    switcher_df = features_df[features_df['id'].isin(SWITCHERS)].copy()

    def overlay_smoothing(ax, x, y, config, groups=None, label_suffix=""):
        """Overlay the smoothing the detectors use, as they use it.

        Detectors smooth WITHIN a session and only over OBSERVED stimuli (NaN
        stimuli -- window edges, stimuli dropped by an RT threshold -- are dropped first and
        inherit a block afterwards). So the curve is computed on the observed values
        of each session and drawn only there: NaN stimuli stay visible as gaps."""
        smoothing_method = config["smoothing_method"]
        smoothing_window = config["smoothing_window"]

        x = np.asarray(x)
        y = np.asarray(y, dtype=float)
        groups = np.zeros(len(y), dtype=int) if groups is None else np.asarray(groups)

        label = (f"Smoothed ({smoothing_method}, w={smoothing_window}, "
                 f"within session){label_suffix}")
        for gi, gval in enumerate(pd.unique(groups)):
            m = (groups == gval) & np.isfinite(y)
            if not m.any():
                continue
            y_s = np.full(len(y), np.nan)
            y_s[m] = apply_smoothing(y[m], smoothing_method, smoothing_window)
            y_s[groups != gval] = np.nan
            ax.plot(x, y_s, 'r-', linewidth=2, alpha=0.9,
                    label=label if gi == 0 else None)


    # Function to visualize blocks for a participant
    def plot_participant_blocks(pid, df, block_methods, switch_type, config=None):
        """Plot block assignments over time for a single participant."""
        pid_df = df[df['id'] == pid].copy()

        if config is None:
            config = {}

        if len(pid_df) == 0:
            print(f"  No data for participant {pid}")
            return None
        
        # Sort by trial order
        if 'stim_order' in pid_df.columns:
            pid_df = pid_df.sort_values(['session', 'stim_order'])
        
        sessions = sorted(pid_df['session'].unique())
        n_methods = len(block_methods)
        
        # Block strips need far less height than the two signal panels, which keeps
        # the figure readable when every detector is plotted.
        fig, axes = plt.subplots(n_methods + 3, 1,
                                 figsize=(14, 9.5 + 1.45 * n_methods), sharex=True,
                                 gridspec_kw={'height_ratios': [2.6, 2.6, 2.6] + [1] * n_methods})
        fig.suptitle(f'Participant {pid} Sessions: {sessions}', fontsize=14, fontweight='bold', y=0.99)
        
        # Create trial index across sessions
        pid_df = pid_df.reset_index(drop=True)
        pid_df['trial_idx'] = range(len(pid_df))
        
        # Mark session boundaries
        session_boundaries = []
        for sess in sessions[1:]:
            first_idx = pid_df[pid_df['session'] == sess]['trial_idx'].min()
            session_boundaries.append(first_idx)
        
        # Panels 1-3: the three LVL_1 detection features, as stored (NaN = gap; this
        # is the diagnostic view -- the publication mosaic bridges the gaps).
        _panels = [(get_feature_column(name, pid_df, config), name, clr)
                   for name, clr in (("Adj RT Level (Stim)", "blue"),
                                     ("Adj RT Var (win)", "darkorange"),
                                     ("Entropy (win)", "green"))]
        for _row, (_col, _name, _clr) in enumerate(_panels):
            ax = axes[_row]
            if _col is not None and _col in pid_df.columns:
                x = pid_df['trial_idx'].values
                y = pid_df[_col].to_numpy(dtype=float)
                ax.plot(x, y, '-', color=_clr, alpha=0.7, linewidth=1, label="Raw (NaN = gap)")
                ax.scatter(x, y, color=_clr, s=10, alpha=0.35)
                # overlay the SAME smoothing the detectors use (per session, observed only); predefined
                # segmentation runs no detector, so there is none to draw
                if not PREDEFINED:
                    overlay_smoothing(ax, x, y, config, groups=pid_df['session'].values)
            ax.set_ylabel(f"{_name}\n({_col})", fontsize=9)
            ax.set_title(f"{_name} over time")
            for boundary in session_boundaries:
                ax.axvline(x=boundary, color='red', linestyle='--', alpha=0.7,
                           label='Session boundary' if _row == 0 else None)
            if _row == 0:
                ax.legend(loc='upper right')
        
        # Plot block assignments for each method
        colors = plt.cm.tab10(np.linspace(0, 1, 10))
        
        for i, method in enumerate(block_methods):
            ax = axes[i + 3]
            # block_<m>_unique is used when present, block_<m> otherwise. `block_<m>`
            # restarts at 0 each session, so session 1 block 0 and session 2 block 0
            # share a colour; the _unique labels (cell 23; dynamic segmentation with
            # controls) do not repeat within a participant.
            col = f'block_{method}_unique'
            if col not in pid_df.columns:
                col = f'block_{method}'
            if col not in pid_df.columns:
                col = method  # Try direct name
            
            if col in pid_df.columns:
                blocks = pid_df[col].values
                unique_blocks = np.unique(blocks[~np.isnan(blocks)])
                
                # Color by block
                for j, block in enumerate(unique_blocks):
                    mask = blocks == block
                    ax.fill_between(pid_df['trial_idx'], 0, 1, where=mask, 
                                   alpha=0.5, color=colors[int(block) % 10], 
                                   label=f'Block {int(block)}')
                
                ax.set_ylabel(method, fontsize=8)
                ax.set_ylim(0, 1)
                ax.set_yticks([])
            
            for boundary in session_boundaries:
                ax.axvline(x=boundary, color='red', linestyle='--', alpha=0.7)
        
        axes[-1].set_xlabel('Trial Index')
        plt.tight_layout()
        return fig

    # Select methods to visualize. None -> every detector, ordered by rank:
    # summary_df (the LVL1 notebook's [cell 24], as exported) is already sorted best-first on the same effect size the
    # method ranking uses, so the panels read top-down from best to worst. Any
    # detector missing from summary_df (e.g. dropped by min_n) is appended after.
    if METHODS is None:
        _avail = {c[len('block_'):] for c in block_cols
                  if not c.startswith('block_Fixed_')}
        _ranked = [m for m in summary_df['method'] if m in _avail]
        select_methods = _ranked + sorted(_avail - set(_ranked))
    else:
        select_methods = METHODS

    # Check which methods are available
    available_methods = [m for m in select_methods if f'block_{m}' in features_df.columns]
    print(f"\nMethods to visualize: {available_methods}")

    # Plot each switcher
    print("\n" + "=" * 80)
    print("VISUALIZING POTENTIAL SWITCHERS")
    print("=" * 80)

    for pid in SWITCHERS:
        print(f"\n--- Participant {pid} ---")
        fig = plot_participant_blocks(pid, features_df, available_methods, "Switcher", config=LVL_1_CONFIG)
        if fig:
            plt.show()